<a href="https://colab.research.google.com/github/sumandash509/AML_LAB/blob/main/AMLLAB08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

calculate karl pearson correlation rx1y,rx2y,rx3y
and rx1x2,rx2x3,rx1x3 after that calculate the CFS merit for candidate subsets.
pipeline
row dataset-pearson correlation rcf,rff
CFS merit
best subset
the dataset is
student= 1 2 3 4 5 6
x1= 1 2 3 7 8 9
x2= 2 3 4 6 7 8
x3= 5 4 3 3 2 1
class (y)= 0 0 0 1 1 1

In [2]:
import numpy as np
from itertools import combinations

# Dataset
x1 = np.array([1, 2, 3, 7, 8, 9])
x2 = np.array([2, 3, 4, 6, 7, 8])
x3 = np.array([5, 4, 3, 3, 2, 1])
y  = np.array([0, 0, 0, 1, 1, 1])

# Features
features = {
    "x1": x1,
    "x2": x2,
    "x3": x3
}

# ------------------------------------------------
# 1. Pearson correlations: feature-class (rcf)
# ------------------------------------------------

print("Feature-Class Correlations:")

rcf = {}

for name, feature in features.items():
    r = np.corrcoef(feature, y)[0, 1]
    rcf[name] = r
    print(f"r_{name}y = {r:.4f}")


# ------------------------------------------------
# 2. Feature-Feature correlations (rff)
# ------------------------------------------------

print("\nFeature-Feature Correlations:")

rff = {}

feature_names = list(features.keys())

for f1, f2 in combinations(feature_names, 2):
    r = np.corrcoef(features[f1], features[f2])[0, 1]
    rff[(f1, f2)] = r
    print(f"r_{f1}{f2} = {r:.4f}")


# ------------------------------------------------
# 3. CFS Merit
# ------------------------------------------------

def cfs_merit(subset):

    k = len(subset)

    # Average feature-class correlation
    avg_rcf = np.mean([rcf[f] for f in subset])

    # Average feature-feature correlation
    if k == 1:
        avg_rff = 0
    else:
        correlations = []

        for f1, f2 in combinations(subset, 2):
            correlations.append(rff[(f1, f2)])

        avg_rff = np.mean(correlations)

    # CFS formula
    merit = (k * avg_rcf) / np.sqrt(
        k + k * (k - 1) * avg_rff
    )

    return merit, avg_rcf, avg_rff


# ------------------------------------------------
# 4. Calculate all candidate subsets
# ------------------------------------------------

results = []

for k in range(1, len(feature_names) + 1):

    for subset in combinations(feature_names, k):

        merit, avg_rcf, avg_rff = cfs_merit(subset)

        results.append((subset, avg_rcf, avg_rff, merit))


# ------------------------------------------------
# 5. Merit Table
# ------------------------------------------------

print("\nCFS Merit Table")
print("-" * 65)
print(f"{'Subset':<20}{'Avg rcf':>12}{'Avg rff':>12}{'Merit':>12}")
print("-" * 65)

for subset, avg_rcf, avg_rff, merit in results:
    subset_name = "{" + ", ".join(subset) + "}"
    print(
        f"{subset_name:<20}"
        f"{avg_rcf:>12.4f}"
        f"{avg_rff:>12.4f}"
        f"{merit:>12.4f}"
    )

print("-" * 65)


# ------------------------------------------------
# 6. Best subset
# ------------------------------------------------

best_subset, _, _, best_merit = max(
    results,
    key=lambda x: x[3]
)

print("\nBest Subset:")
print(best_subset)
print(f"Best CFS Merit = {best_merit:.4f}")


Feature-Class Correlations:
r_x1y = 0.9649
r_x2y = 0.9258
r_x3y = -0.7746

Feature-Feature Correlations:
r_x1x2 = 0.9926
r_x1x3 = -0.9135
r_x2x3 = -0.9562

CFS Merit Table
-----------------------------------------------------------------
Subset                   Avg rcf     Avg rff       Merit
-----------------------------------------------------------------
{x1}                      0.9649      0.0000      0.9649
{x2}                      0.9258      0.0000      0.9258
{x3}                     -0.7746      0.0000     -0.7746
{x1, x2}                  0.9454      0.9926      0.9471
{x1, x3}                  0.0952     -0.9135      0.4575
{x2, x3}                  0.0756     -0.9562      0.5108
{x1, x2, x3}              0.3720     -0.2924      1.0000
-----------------------------------------------------------------

Best Subset:
('x1', 'x2', 'x3')
Best CFS Merit = 1.0000
